# Stage 2 — Unified Summary

Compares all Stage 2 variants on a common 30-image representative set.  
Rank-orders methods by median L2 improvement at three query budgets (200 / 500 / 1000).  
Each variant is the best configuration from its respective experiment notebook.

## Variants compared

| Variant | Notebook | Description |
|---|---|---|
| Stage 1 baseline | — | sep-CMA-ES λ=10, k=40, 1000q, no restart |
| Stag restart T=15 | exp1 | Stage 1 + stagnation-triggered family restart |
| IPOP T=15 | exp1 | Stage 1 + IPOP (λ doubles on stagnation) |
| (1+1)-sep | exp3 | Diagonal (1+1)-CMA-ES |
| (1+1)-full | exp3 | Full k×k (1+1)-CMA-ES |
| Best (k,λ) | exp2 | Best (k,λ) configuration from the grid |
| DCT+SLIC | exp5 | Image-adaptive SLIC superpixel basis |
| Full-CMA-ES λ=10 | exp4 | Full k×k covariance CMA-ES |

All at Phase-3 budget 1000 queries unless stated.

In [1]:
import sys, os
from pathlib import Path

_candidates = [Path.cwd(), Path.cwd()/'STAGE_2', Path.cwd()/'Adversial ML'/'STAGE_2']
for _p in _candidates:
    if (_p / 'utils_stage2.py').exists():
        sys.path.insert(0, str(_p.resolve())); break
else:
    raise FileNotFoundError('utils_stage2.py not found')

import numpy as np
import torch
import matplotlib.pyplot as plt
from skimage.segmentation import slic as skimage_slic

from utils_stage2 import (
    clip01, compute_ssim, compute_l2, compute_linf,
    Oracle, load_models, get_jointly_correct,
    phase1, phase2, build_subspace_with_dc, build_dct_basis, build_grid_basis,
    sep_cmaes, lam_default,
    SSIM_STOP, H, W, C, RANDOM_SEED,
)

np.random.seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

N_SUMMARY = 30
P3_BUDGET = 1000
BUDGETS   = [200, 500, 1000]
os.makedirs('outputs/summary', exist_ok=True)

# ---------------------------------------------------------------------------
# Variant functions (inline — cannot import from .ipynb files)
# ---------------------------------------------------------------------------

def one_plus_one_sep(oracle, img_hwc, x_adv_init, B, max_queries,
                      ssim_stop=SSIM_STOP):
    """
    (1+1)-CMA-ES diagonal. Sigma driven by is_adv (feasibility signal),
    covariance/mean updated only on strict L2 improvement.
    """
    k = B.shape[0]; x0 = img_hwc.flatten()
    delta0 = x_adv_init.flatten() - x0
    theta_m = (B @ delta0).astype(np.float32)
    cc = 2.0/(k+2); c1 = 2.0/(k**2+6)
    c_p = 1.0/(5+k**0.5/2); d_damp = 1.0+k/2
    p_target = 0.2; p_succ = p_target
    sigma = float(max(0.1*np.linalg.norm(delta0)/np.sqrt(k), 1e-5))
    D = np.ones(k, dtype=np.float32); pc = np.zeros(k, dtype=np.float32)
    best_x = x_adv_init.copy(); best_l2 = compute_l2(img_hwc, best_x)
    best_ssim = compute_ssim(img_hwc, best_x); history, queries = [], 0
    while queries < max_queries:
        z = np.random.randn(k).astype(np.float32)
        theta_new = theta_m + sigma*D*z
        x_cand = clip01(x0 + B.T@theta_new).reshape(H, W, C)
        is_adv, _ = oracle.query(x_cand, phase='p3'); queries += 1
        l2_cand = compute_l2(img_hwc, x_cand)
        improves = is_adv and l2_cand < best_l2
        if improves:
            theta_m = theta_new; best_l2 = l2_cand; best_x = x_cand.copy()
            best_ssim = compute_ssim(img_hwc, best_x)
            step = D * z
            pc = (1-cc)*pc + np.sqrt(cc*(2-cc))*step
            D = np.sqrt(np.clip((1-c1)*D**2 + c1*pc**2, 1e-20, 1e10))
        p_succ = (1-c_p)*p_succ + c_p*float(is_adv)  # feasibility drives sigma
        sigma = float(np.clip(sigma*np.exp((p_succ-p_target)/d_damp), 1e-10, 10.0))
        history.append({'queries': queries, 'best_l2': best_l2, 'best_ssim': best_ssim,
                        'sigma': sigma, 'success': improves})
        if best_ssim >= ssim_stop: break
    return best_x, best_l2, best_ssim, queries, history


def one_plus_one_full(oracle, img_hwc, x_adv_init, B, max_queries,
                       ssim_stop=SSIM_STOP):
    """
    (1+1)-CMA-ES full covariance. Sigma driven by is_adv (feasibility signal).
    Uses 'cov' not 'C' to avoid shadowing C=3 channel constant.
    """
    k = B.shape[0]; x0 = img_hwc.flatten()
    delta0 = x_adv_init.flatten() - x0
    theta_m = (B @ delta0).astype(np.float64)
    cc = 2.0/(k+2); c1 = 2.0/(k**2+6)
    c_p = 1.0/(5+k**0.5/2); d_damp = 1.0+k/2
    p_target = 0.2; p_succ = p_target
    sigma = float(max(0.1*np.linalg.norm(delta0)/np.sqrt(k), 1e-5))
    cov = np.eye(k, dtype=np.float64); pc = np.zeros(k, dtype=np.float64)
    best_x = x_adv_init.copy(); best_l2 = compute_l2(img_hwc, best_x)
    best_ssim = compute_ssim(img_hwc, best_x); history, queries = [], 0
    while queries < max_queries:
        try: L = np.linalg.cholesky(cov + 1e-12*np.eye(k))
        except np.linalg.LinAlgError:
            cov = np.eye(k, dtype=np.float64); L = np.eye(k, dtype=np.float64)
        z = np.random.randn(k); y = L @ z
        theta_new = (theta_m + sigma*y).astype(np.float32)
        x_cand = clip01(x0 + B.T@theta_new).reshape(H, W, C)
        is_adv, _ = oracle.query(x_cand, phase='p3'); queries += 1
        l2_cand = compute_l2(img_hwc, x_cand)
        improves = is_adv and l2_cand < best_l2
        if improves:
            theta_m = theta_new.astype(np.float64); best_l2 = l2_cand
            best_x = x_cand.copy(); best_ssim = compute_ssim(img_hwc, best_x)
            pc = (1-cc)*pc + np.sqrt(cc*(2-cc))*y
            cov = (1-c1)*cov + c1*np.outer(pc, pc); cov = 0.5*(cov+cov.T)
        p_succ = (1-c_p)*p_succ + c_p*float(is_adv)  # feasibility drives sigma
        sigma = float(np.clip(sigma*np.exp((p_succ-p_target)/d_damp), 1e-10, 10.0))
        history.append({'queries': queries, 'best_l2': best_l2, 'best_ssim': best_ssim,
                        'sigma': sigma, 'success': improves})
        if best_ssim >= ssim_stop: break
    return best_x, best_l2, best_ssim, queries, history


def full_cmaes(oracle, img_hwc, x_adv_init, B, lam, max_queries,
               ssim_stop=SSIM_STOP):
    """Full k×k CMA-ES. Uses 'Cov' not 'C'. No L-inf gate."""
    k = B.shape[0]; x0 = img_hwc.flatten()
    delta0 = x_adv_init.flatten() - x0; theta_m = (B @ delta0).astype(np.float64)
    mu = max(1, lam//2)
    w_raw = np.log(mu+0.5) - np.log(np.arange(1, mu+1, dtype=np.float64))
    w = w_raw/w_raw.sum(); mueff = 1.0/np.sum(w**2)
    cc = (4+mueff/k)/(k+4+2*mueff/k); cs = (mueff+2)/(k+mueff+5)
    c1 = 2.0/((k+1.3)**2+mueff)
    cmu = min(1-c1, 2*(mueff-2+1/mueff)/((k+2)**2+mueff))
    damps = 1+2*max(0.0, np.sqrt((mueff-1)/(k+1))-1)+cs
    chiN = k**0.5*(1-1/(4*k)+1/(21*k**2))
    Cov = np.eye(k, dtype=np.float64); pc = np.zeros(k); ps = np.zeros(k)
    sigma = float(max(0.1*np.linalg.norm(delta0)/np.sqrt(k), 1e-5))
    best_x = x_adv_init.copy(); best_l2 = compute_l2(img_hwc, best_x)
    best_ssim = compute_ssim(img_hwc, best_x); history, gen, queries = [], 0, 0
    while queries < max_queries:
        batch = min(lam, max_queries-queries)
        if batch <= 0: break
        evals, Q = np.linalg.eigh(Cov); evals = np.maximum(evals, 1e-20)
        zs = np.random.randn(batch, k)
        ys = (Q * np.sqrt(evals)[None, :]) @ zs.T; ys = ys.T
        thetas = theta_m[None, :] + sigma*ys
        pool = []
        for i in range(batch):
            x_cand = clip01(x0 + B.T@thetas[i].astype(np.float32)).reshape(H, W, C)
            is_adv, _ = oracle.query(x_cand, phase='p3'); queries += 1
            if is_adv:
                l2 = compute_l2(img_hwc, x_cand); pool.append((l2, zs[i], ys[i], x_cand))
                if l2 < best_l2: best_l2=l2; best_x=x_cand.copy(); best_ssim=compute_ssim(img_hwc, best_x)
        history.append({'gen':gen, 'queries':queries, 'best_l2':best_l2,
                        'best_ssim':best_ssim, 'sigma':sigma})
        if best_ssim >= ssim_stop: break
        if not pool: sigma=min(sigma*2.0, 10.0); gen+=1; continue
        pool.sort(key=lambda t: t[0]); sel=pool[:mu]; n_sel=len(sel)
        w_s = w[:n_sel]/w[:n_sel].sum()
        ys_sel=np.array([t[2] for t in sel]); zs_sel=np.array([t[1] for t in sel])
        yw=(w_s[:,None]*ys_sel).sum(0); zw=(w_s[:,None]*zs_sel).sum(0)
        theta_m = theta_m + sigma*yw
        ps=(1-cs)*ps+np.sqrt(cs*(2-cs)*mueff)*zw; gen+=1
        hsig=(np.linalg.norm(ps)/chiN/np.sqrt(1-(1-cs)**(2*gen)))<(1.4+2/(k+1))
        pc=(1-cc)*pc+hsig*np.sqrt(cc*(2-cc)*mueff)*yw
        rank_mu=sum(w_s[i]*np.outer(ys_sel[i],ys_sel[i]) for i in range(n_sel))
        Cov=(1-c1-cmu)*Cov+c1*(np.outer(pc,pc)+(1-hsig)*cc*(2-cc)*Cov)+cmu*rank_mu
        Cov=0.5*(Cov+Cov.T)
        sigma=float(np.clip(sigma*np.exp((cs/damps)*(np.linalg.norm(ps)/chiN-1)), 1e-10, 10.0))
    return best_x, best_l2, best_ssim, queries, history


def build_slic_basis(img_hwc, k_dct=20, k_sp=20, compactness=10.0):
    """SLIC superpixel + DC-inclusive DCT basis."""
    segments = skimage_slic(img_hwc, n_segments=k_sp, compactness=compactness,
                             channel_axis=2, start_label=0, convert2lab=True)
    K_actual = segments.max() + 1
    sp_vecs = []
    for seg_id in range(K_actual):
        mask = (segments == seg_id)
        if mask.sum() == 0: continue
        v = np.zeros((H, W, C), dtype=np.float32); v[mask, :] = 1.0
        v_flat = v.flatten(); nrm = np.linalg.norm(v_flat)
        if nrm > 1e-12: sp_vecs.append(v_flat / nrm)
    B_sp  = np.array(sp_vecs[:k_sp], dtype=np.float32)
    B_dct = build_dct_basis(k_dct=k_dct, H=H, W=W, C=C)
    B_raw = np.vstack([B_dct, B_sp]).astype(np.float64)
    Q, _  = np.linalg.qr(B_raw.T)
    return Q.T[:B_raw.shape[0]].astype(np.float32), segments

Device: cpu


In [2]:
model_std, model_rob = load_models(device)
images = get_jointly_correct(model_std, model_rob, device, n=N_SUMMARY)
B_grid = build_subspace_with_dc(k_dct=20, k_sp=20)   # DC-inclusive basis
k = B_grid.shape[0]
print(f'Images: {len(images)}   k={k}')

Images: 30   k=40


## Stagnation restart helper (from exp1)

In [3]:
def stagnation_restart(oracle, img_hwc, x_bnd_init, winner_init, B,
                        stag_T=15, total_budget=1000):
    budget_first = total_budget // 2
    best_x, best_l2, best_ssim, q1, hist1, stagnated = sep_cmaes(
        oracle, img_hwc, x_bnd_init, B,
        lam=10, max_queries=budget_first, stag_T=stag_T)
    histories = [('run-1', hist1)]
    remaining = total_budget - q1
    if not stagnated or remaining < 10:
        return best_x, best_l2, best_ssim, q1, histories
    x_bnd2, winner2 = phase1(oracle, img_hwc, seed=42, exclude={winner_init})
    if x_bnd2 is None:
        bx2, bl2, bss, q2, h2 = sep_cmaes(
            oracle, img_hwc, x_bnd_init, B, lam=10, max_queries=remaining)
    else:
        x_bnd2 = phase2(oracle, img_hwc, x_bnd2)
        bx2, bl2, bss, q2, h2 = sep_cmaes(
            oracle, img_hwc, x_bnd2, B, lam=10, max_queries=remaining)
    histories.append(('run-2', h2))
    if bl2 < best_l2:
        best_l2, best_ssim = bl2, bss
    all_hist = [{'queries': h['queries'], 'best_l2': best_l2, 'best_ssim': best_ssim}
                for h in hist1]
    offset = q1
    for h in h2:
        all_hist.append({'queries': offset + h['queries'], 'best_l2': min(best_l2, h['best_l2']),
                         'best_ssim': h['best_ssim']})
    return best_x, best_l2, best_ssim, q1+q2, all_hist


def ipop_restart(oracle, img_hwc, x_bnd_init, B,
                  stag_T=15, total_budget=1000):
    lam = 4
    lam_max = k
    queries_total = 0
    best_x = x_bnd_init.copy()
    best_l2 = compute_l2(img_hwc, best_x)
    best_ssim = compute_ssim(img_hwc, best_x)
    all_hist = []
    x_restart = x_bnd_init
    restart = 0
    while queries_total < total_budget:
        rem = total_budget - queries_total
        if rem < lam:
            break
        bx, bl, bss, q, hist, stag = sep_cmaes(
            oracle, img_hwc, x_restart, B,
            lam=lam, max_queries=rem, stag_T=stag_T)
        queries_total += q
        for h in hist:
            all_hist.append({'queries': queries_total - q + h['queries'],
                             'best_l2': h['best_l2'], 'best_ssim': h['best_ssim']})
        if bl < best_l2:
            best_l2, best_x, best_ssim = bl, bx, bss
        if best_ssim >= SSIM_STOP or not stag:
            break
        lam = min(lam_max, lam * 2)
        x_restart = best_x
        restart += 1
    return best_x, best_l2, best_ssim, queries_total, all_hist

## Define all variants

In [4]:
def make_variant(fn_name):
    """
    Each variant receives (oracle, img_hwc, x_bnd, winner, B) and returns
    (best_x, best_l2, best_ssim, queries, history).
    """
    def baseline(o, img, xb, wn, B_):       return sep_cmaes(o, img, xb, B_, lam=10, max_queries=P3_BUDGET)
    def stag_restart(o, img, xb, wn, B_):   return stagnation_restart(o, img, xb, wn, B_, stag_T=15, total_budget=P3_BUDGET)
    def ipop(o, img, xb, wn, B_):           return ipop_restart(o, img, xb, B_, stag_T=15, total_budget=P3_BUDGET)
    def v_1p1_sep(o, img, xb, wn, B_):      return one_plus_one_sep(o, img, xb, B_, max_queries=P3_BUDGET)
    def v_1p1_full(o, img, xb, wn, B_):     return one_plus_one_full(o, img, xb, B_, max_queries=P3_BUDGET)
    def v_full_cmaes(o, img, xb, wn, B_):   return full_cmaes(o, img, xb, B_, lam=10, max_queries=P3_BUDGET)
    def v_slic(o, img, xb, wn, B_):
        B_s, _ = build_slic_basis(img, k_dct=20, k_sp=20)
        return sep_cmaes(o, img, xb, B_s, lam=10, max_queries=P3_BUDGET)

    lookup = {
        'baseline':       baseline,
        'stag_restart':   stag_restart,
        'ipop':           ipop,
        '(1+1)-sep':      v_1p1_sep,
        '(1+1)-full':     v_1p1_full,
        'full-CMA-ES':    v_full_cmaes,
        'DCT+SLIC':       v_slic,
    }
    return lookup[fn_name]

VARIANT_NAMES = ['baseline', 'stag_restart', 'ipop',
                 '(1+1)-sep', '(1+1)-full', 'full-CMA-ES', 'DCT+SLIC']
VARIANTS = {name: make_variant(name) for name in VARIANT_NAMES}

## Run all variants on both models

In [5]:
MODELS = [('standard', model_std), ('robust', model_rob)]

all_results  = {v: {m: [] for m, _ in MODELS} for v in VARIANT_NAMES}
all_histories = {v: {m: [] for m, _ in MODELS} for v in VARIANT_NAMES}

# Precompute Phase 1+2 per model
p12 = {m: {} for m, _ in MODELS}
for model_name, model in MODELS:
    for rec in images:
        o = Oracle(model, rec['label'], device)
        xb, wn = phase1(o, rec['img'], seed=rec['idx'])
        if xb is None:
            p12[model_name][rec['idx']] = None
        else:
            xb = phase2(o, rec['img'], xb)
            p12[model_name][rec['idx']] = (xb, wn)
print('Phase 1+2 precomputed.')

Phase 1+2 precomputed.


In [ ]:
for model_name, model in MODELS:
    print(f'\n=== {model_name.upper()} ===')
    for vname, vfn in VARIANTS.items():
        for rec in images:
            entry = p12[model_name].get(rec['idx'])
            if entry is None:
                all_results[vname][model_name].append(None)
                all_histories[vname][model_name].append([])
                continue
            xb, wn = entry
            l2_p2 = compute_l2(rec['img'], xb)
            o = Oracle(model, rec['label'], device)
            out = vfn(o, rec['img'], xb, wn, B_grid)
            # out: (best_x, best_l2, best_ssim, queries, history)
            all_results[vname][model_name].append(
                dict(l2_p2=l2_p2, l2_p3=out[1], ssim=out[2],
                     improvement=l2_p2-out[1], q=out[3]))
            all_histories[vname][model_name].append(out[4])

        valid = [r for r in all_results[vname][model_name] if r]
        med = np.median([r['l2_p3'] for r in valid])
        imp = np.median([r['improvement'] for r in valid])
        print(f'  {vname:<22}: med_L2={med:.4f}  med_imp={imp:.4f}')

## Summary table — L2 at 200 / 500 / 1000 queries

In [ ]:
def l2_at_budget(hist, budget):
    if not hist:
        return None
    qs  = np.array([h['queries']  for h in hist])
    l2s = np.array([h['best_l2'] for h in hist])
    return float(np.interp(budget, qs, l2s))

for model_name, _ in MODELS:
    print(f'\n=== {model_name.upper()} — Median L2 at budget ===')
    header = f'{"Variant":<22} {"200q":>8} {"500q":>8} {"1000q":>8} {"SSIM":>8}'
    print(header)
    print('-' * len(header))
    for vname in VARIANT_NAMES:
        hists = all_histories[vname][model_name]
        recs  = all_results[vname][model_name]
        l2_200 = np.median([l2_at_budget(h, 200)  for h in hists if h and l2_at_budget(h, 200)])
        l2_500 = np.median([l2_at_budget(h, 500)  for h in hists if h and l2_at_budget(h, 500)])
        l2_1000= np.median([l2_at_budget(h, 1000) for h in hists if h and l2_at_budget(h, 1000)])
        ssim   = np.median([r['ssim'] for r in recs if r])
        print(f'{vname:<22} {l2_200:>8.4f} {l2_500:>8.4f} {l2_1000:>8.4f} {ssim:>8.4f}')

## Plot 1 — L2 curves: all variants (standard model)

In [ ]:
COLORS = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B3','#937860','#DA8BC3']
STYLES = ['-', '-', '--', '-.', ':', '-', '--']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (model_name, _) in zip(axes, MODELS):
    for (vname, color, ls) in zip(VARIANT_NAMES, COLORS, STYLES):
        hists = [h for h in all_histories[vname][model_name] if h]
        if not hists:
            continue
        q_max  = max(h[-1]['queries'] for h in hists)
        q_grid = np.arange(1, q_max + 1)
        curves = []
        for hist in hists:
            qs  = np.array([h['queries']  for h in hist])
            l2s = np.array([h['best_l2'] for h in hist])
            curves.append(np.interp(q_grid, qs, l2s))
        arr = np.array(curves)
        med = np.median(arr, axis=0)
        ax.plot(q_grid, med, label=vname, color=color, ls=ls)
        p25 = np.percentile(arr, 25, axis=0)
        p75 = np.percentile(arr, 75, axis=0)
        ax.fill_between(q_grid, p25, p75, alpha=0.06, color=color)

    for bm in BUDGETS:
        ax.axvline(bm, color='gray', ls=':', lw=0.7, alpha=0.5)
    ax.set_xlabel('Phase-3 queries')
    ax.set_ylabel('Best L2 (median)')
    ax.set_title(f'{model_name}')
    ax.legend(fontsize=7)

plt.suptitle('Stage 2 Summary — All variants vs Stage 1 baseline', fontsize=11)
plt.tight_layout()
plt.savefig('outputs/summary/l2_curves_all.png', dpi=150)
plt.show()

## Plot 2 — Ranking bar chart at 500 queries

In [ ]:
TARGET_BUDGET = 500

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (model_name, _) in zip(axes, MODELS):
    ranking = []
    for vname in VARIANT_NAMES:
        hists = all_histories[vname][model_name]
        vals  = [l2_at_budget(h, TARGET_BUDGET) for h in hists
                 if h and l2_at_budget(h, TARGET_BUDGET) is not None]
        if vals:
            ranking.append((vname, np.median(vals)))
    ranking.sort(key=lambda t: t[1])

    names = [r[0] for r in ranking]
    vals  = [r[1] for r in ranking]
    colors_bar = ['#55A868' if n == 'baseline' else '#4C72B0' for n in names]
    bars = ax.barh(names, vals, color=colors_bar, edgecolor='k', alpha=0.85)
    ax.set_xlabel(f'Median L2 at {TARGET_BUDGET} Phase-3 queries (lower = better)')
    ax.set_title(f'{model_name}')
    for bar, val in zip(bars, vals):
        ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)

plt.suptitle(f'Stage 2 Summary — Method ranking at {TARGET_BUDGET} queries', fontsize=11)
plt.tight_layout()
plt.savefig('outputs/summary/ranking_bar.png', dpi=150)
plt.show()

## Plot 3 — Qualitative examples: best adversarial images per method

In [ ]:
# Show one representative image: original, Phase-2 boundary, then each variant's best
model_name = 'standard'
# Pick the image where baseline produced the worst result (hardest image)
valid_baseline = [(i, r) for i, r in enumerate(all_results['baseline'][model_name]) if r]
worst_idx = max(valid_baseline, key=lambda t: t[1]['l2_p3'])[0]
rec = images[worst_idx]

entry = p12[model_name].get(rec['idx'])
if entry is not None:
    xb, _ = entry
    n_variants = len(VARIANT_NAMES)
    fig, axes = plt.subplots(1, n_variants + 2, figsize=(2.5*(n_variants+2), 3))

    axes[0].imshow(np.clip(rec['img'], 0, 1))
    axes[0].set_title('original', fontsize=8)
    axes[0].axis('off')

    axes[1].imshow(np.clip(xb, 0, 1))
    l2_bnd = compute_l2(rec['img'], xb)
    axes[1].set_title(f'Phase 2 boundary\nL2={l2_bnd:.2f}', fontsize=8)
    axes[1].axis('off')

    for ax, vname in zip(axes[2:], VARIANT_NAMES):
        r = all_results[vname][model_name][worst_idx]
        if r is None:
            ax.axis('off')
            continue
        # Re-run to get actual best_x image
        o = Oracle(model_std, rec['label'], device)
        wn = p12[model_name][rec['idx']][1]
        out = VARIANTS[vname](o, rec['img'], xb, wn, B_grid)
        ax.imshow(np.clip(out[0], 0, 1))
        ax.set_title(f'{vname}\nL2={out[1]:.2f}\nSSIM={out[2]:.3f}', fontsize=7)
        ax.axis('off')

    plt.suptitle(f'Stage 2 — Adversarial images (standard model, hardest image)', fontsize=10)
    plt.tight_layout()
    plt.savefig('outputs/summary/qualitative_examples.png', dpi=150)
    plt.show()

print('Stage 2 summary complete. Outputs saved to outputs/summary/')
print('Key recommendation: see ranking bar chart for best method at your query budget.')